In [18]:
import torch
from torch_geometric.utils import to_networkx
import networkx as nx
import matplotlib.pyplot as plt

In [19]:
# Load only one graph (index 0), not the full merged dataset.
import torch
from torch_geometric.data.separate import separate

loaded = torch.load(
    '/Users/anshumaansoni/PycharmProjects/Molecular-Property-Prediction-Using-GNN-and-Transformer/data/processed/qm_merged_3d_graphs.pt',
    map_location=torch.device('cpu'),
    weights_only=False,
 )

if isinstance(loaded, tuple) and len(loaded) >= 2 and isinstance(loaded[1], dict):
    # InMemoryDataset serialized layout: (collated_data, slices)
    collated_data, slices = loaded[0], loaded[1]
    data = separate(
        cls=collated_data.__class__,
        batch=collated_data,
        idx=0,
        slice_dict=slices,
        decrement=False,
    )
elif isinstance(loaded, list):
    data = loaded[0]
elif hasattr(loaded, 'to_data_list'):
    data = loaded.to_data_list()[0]
else:
    data = loaded

In [20]:
from torch_geometric.utils import to_networkx
G = to_networkx(data, to_undirected=True, node_attrs=['x'], edge_attrs=['edge_attr'])

In [ ]:
# Class-based interactive 3D molecular graph plotter (Plotly).
import numpy as np
import torch
from collections import Counter
from torch_geometric.data.separate import separate

try:
    import plotly.graph_objects as go
except ImportError as exc:
    raise ImportError('Plotly is not installed. Run: pip install plotly') from exc


class MoleculeInteractivePlotter:
    def __init__(self, graph_info=None, data=None, data_path=None):
        # graph_info lets you provide raw graph tensors/arrays directly.
        self.graph_info = graph_info
        self.data = data
        self.data_path = data_path or (
            '/Users/anshumaansoni/PycharmProjects/Molecular-Property-Prediction-Using-GNN-and-Transformer/data/processed/qm_merged_3d_graphs.pt'
        )

        # Shared visual mappings.
        self.atom_color = {
            0: '#F8FAFC', 1: '#334155', 2: '#2563EB', 3: '#DC2626', 4: '#14B8A6',
            5: '#8B5CF6', 6: '#334155', 7: '#2563EB', 8: '#DC2626', 9: '#14B8A6',
        }
        self.atom_size = {0: 11, 1: 17, 2: 18, 3: 19, 4: 20, 6: 17, 7: 18, 8: 19, 9: 20}
        self.edge_type_name = {1: 'Single', 2: 'Double', 3: 'Triple', 4: 'Aromatic', 0: 'Other'}
        self.edge_type_color = {1: '#475569', 2: '#0EA5E9', 3: '#8B5CF6', 4: '#F59E0B', 0: '#64748B'}
        self.edge_type_width = {1: 4.0, 2: 5.5, 3: 7.0, 4: 4.8, 0: 3.5}

    @staticmethod
    def _to_numpy(arr):
        if arr is None:
            return None
        if isinstance(arr, np.ndarray):
            return arr
        if torch.is_tensor(arr):
            return arr.detach().cpu().numpy()
        return np.asarray(arr)

    def _load_first_graph(self):
        if self.data is not None:
            return self.data

        loaded = torch.load(
            self.data_path,
            map_location=torch.device('cpu'),
            weights_only=False,
        )
        if isinstance(loaded, tuple) and len(loaded) >= 2 and isinstance(loaded[1], dict):
            collated_data, slices = loaded[0], loaded[1]
            return separate(
                cls=collated_data.__class__,
                batch=collated_data,
                idx=0,
                slice_dict=slices,
                decrement=False,
            )
        if isinstance(loaded, list):
            return loaded[0]
        if hasattr(loaded, 'to_data_list'):
            return loaded.to_data_list()[0]
        return loaded

    def _extract_arrays(self):
        # Preferred mode: user-provided graph_info.
        if self.graph_info is not None:
            pos = self._to_numpy(self.graph_info.get('pos'))
            edge_index = self._to_numpy(self.graph_info.get('edge_index'))
            x = self._to_numpy(self.graph_info.get('x'))
            edge_attr = self._to_numpy(self.graph_info.get('edge_attr'))

            if pos is None or pos.ndim != 2 or pos.shape[1] < 3:
                raise ValueError("graph_info['pos'] must be shape [num_nodes, >=3].")
            if edge_index is None or edge_index.ndim != 2 or edge_index.shape[0] != 2:
                raise ValueError("graph_info['edge_index'] must be shape [2, num_edges].")

            return pos[:, :3], edge_index, x, edge_attr

        # Fallback mode: use PyG Data object.
        data = self._load_first_graph()
        if not (hasattr(data, 'pos') and data.pos is not None and data.pos.size(1) >= 3):
            raise ValueError('This graph does not contain valid 3D coordinates in data.pos.')

        xyz = data.pos[:, :3].detach().cpu().numpy()
        edge_index = data.edge_index.detach().cpu().numpy() if hasattr(data, 'edge_index') and data.edge_index is not None else np.zeros((2, 0), dtype=int)
        x = data.x.detach().cpu().numpy() if hasattr(data, 'x') and data.x is not None else None
        edge_attr = data.edge_attr.detach().cpu().numpy() if hasattr(data, 'edge_attr') and data.edge_attr is not None else None
        return xyz, edge_index, x, edge_attr

    def _get_atom_ids_and_labels(self, x, num_nodes):
        if x is not None:
            if x.ndim == 2 and x.shape[1] > 1 and x.min() >= 0 and x.max() <= 1.0 + 1e-6:
                atom_id = np.argmax(x, axis=1)
            else:
                atom_id = np.rint(x[:, 0]).astype(int)
        else:
            atom_id = np.zeros(num_nodes, dtype=int)

        if atom_id.max() <= 4:
            atom_label = {0: 'H', 1: 'C', 2: 'N', 3: 'O', 4: 'F'}
        else:
            atom_label = {0: 'X', 1: 'H', 6: 'C', 7: 'N', 8: 'O', 9: 'F'}
        return atom_id, atom_label

    def _infer_bond_type(self, feat_vec, p_i, p_j):
        if feat_vec is not None:
            f = np.atleast_1d(feat_vec).astype(float)
            if f.shape[0] >= 4:
                head = f[:4]
                if np.all(head >= 0) and float(np.sum(head)) > 0.5:
                    return int(np.argmax(head)) + 1
            scalar = float(f[0])
            scalar_i = int(np.rint(scalar))
            if 1 <= scalar_i <= 4 and abs(scalar - scalar_i) < 0.25:
                return scalar_i

        # Geometric fallback used only for visualization.
        d = float(np.linalg.norm(p_i - p_j))
        if d < 1.27:
            return 3
        if d < 1.39:
            return 2
        if d < 1.52:
            return 4
        return 1

    def _build_edge_meta(self, xyz, edge_index, edge_attr):
        edge_records = {}
        for k, (s, t) in enumerate(edge_index.T):
            a, b = int(s), int(t)
            if a == b:
                continue
            key = (a, b) if a < b else (b, a)
            feat = edge_attr[k] if edge_attr is not None and k < len(edge_attr) else None
            edge_records.setdefault(key, []).append(feat)

        edge_meta = []
        for i, j in sorted(edge_records.keys()):
            feats = [f for f in edge_records[(i, j)] if f is not None]
            feat_avg = np.mean(np.vstack(feats), axis=0) if feats else None
            btype = self._infer_bond_type(feat_avg, xyz[i], xyz[j])
            edge_meta.append((i, j, btype))
        return edge_meta

    def create_figure(self):
        xyz, edge_index, x, edge_attr = self._extract_arrays()
        atom_id, atom_label = self._get_atom_ids_and_labels(x, xyz.shape[0])

        node_colors = [self.atom_color.get(int(v), '#6B7280') for v in atom_id]
        node_sizes = [self.atom_size.get(int(v), 14) for v in atom_id]
        node_labels = [atom_label.get(int(v), f'T{int(v)}') for v in atom_id]

        edge_meta = self._build_edge_meta(xyz, edge_index, edge_attr)
        fig = go.Figure()

        # One trace per bond type for legend-based toggling.
        edge_types = sorted(set(b for _, _, b in edge_meta)) if edge_meta else [0]
        for t in edge_types:
            xs, ys, zs, hover = [], [], [], []
            for i, j, btype in edge_meta:
                if btype != t:
                    continue
                xs += [xyz[i, 0], xyz[j, 0], None]
                ys += [xyz[i, 1], xyz[j, 1], None]
                zs += [xyz[i, 2], xyz[j, 2], None]
                dist = float(np.linalg.norm(xyz[i] - xyz[j]))
                text = f'Bond: {self.edge_type_name.get(t, f"Type {t}")}<br>{i} - {j}<br>Length: {dist:.3f}'
                hover += [text, text, '']
            if xs:
                fig.add_trace(go.Scatter3d(
                    x=xs, y=ys, z=zs,
                    mode='lines',
                    name=f"Bond: {self.edge_type_name.get(t, f'Type {t}')}",
                    line=dict(
                        color=self.edge_type_color.get(t, '#64748B'),
                        width=self.edge_type_width.get(t, 3.5),
                    ),
                    hovertext=hover,
                    hoverinfo='text',
                    opacity=0.95,
                ))

        node_hover = [
            f"Node {i}<br>Atom: {node_labels[i]}<br>x: {xyz[i,0]:.3f}<br>y: {xyz[i,1]:.3f}<br>z: {xyz[i,2]:.3f}"
            for i in range(xyz.shape[0])
        ]
        fig.add_trace(go.Scatter3d(
            x=xyz[:, 0],
            y=xyz[:, 1],
            z=xyz[:, 2],
            mode='markers+text',
            text=[f"{node_labels[i]}{i}" for i in range(xyz.shape[0])],
            textposition='top center',
            textfont=dict(size=10, color='#111827'),
            name='Atoms',
            marker=dict(
                size=node_sizes,
                color=node_colors,
                line=dict(color='#0B1220', width=1),
                opacity=0.98,
            ),
            hovertext=node_hover,
            hoverinfo='text',
        ))

        atom_counts = Counter(int(v) for v in atom_id)
        edge_counts = Counter(int(b) for _, _, b in edge_meta)
        avg_bond = float(np.mean([np.linalg.norm(xyz[i] - xyz[j]) for i, j, _ in edge_meta])) if edge_meta else 0.0

        stats_lines = [
            f"Nodes: {xyz.shape[0]}",
            f"Edges: {len(edge_meta)}",
            f"Avg bond length: {avg_bond:.3f}",
            "",
            "Atom counts:",
        ]
        for k in sorted(atom_counts):
            stats_lines.append(f"{atom_label.get(k, f'T{k}')}: {atom_counts[k]}")
        stats_lines.append('')
        stats_lines.append('Edge counts:')
        for k in sorted(edge_counts):
            stats_lines.append(f"{self.edge_type_name.get(k, f'Type {k}')}: {edge_counts[k]}")

        fig.update_layout(
            title='Interactive Single Molecule Graph (3D) - Class Based',
            template='plotly_white',
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                aspectmode='data',
                bgcolor='rgba(238,242,247,1)',
                camera=dict(eye=dict(x=1.55, y=1.35, z=1.1)),
            ),
            paper_bgcolor='rgba(238,242,247,1)',
            plot_bgcolor='rgba(238,242,247,1)',
            legend=dict(
                x=0.01, y=0.99,
                bgcolor='rgba(255,255,255,0.9)',
                bordercolor='#CBD5E1',
                borderwidth=1,
            ),
            margin=dict(l=0, r=0, t=60, b=0),
            annotations=[dict(
                text='<br>'.join(stats_lines),
                x=0.99, y=0.02,
                xref='paper', yref='paper',
                xanchor='right', yanchor='bottom',
                showarrow=False,
                align='left',
                bordercolor='#CBD5E1',
                borderwidth=1,
                borderpad=6,
                bgcolor='rgba(255,255,255,0.94)',
                font=dict(size=12, color='#0F172A'),
            )],
        )
        return fig

    def show(self):
        fig = self.create_figure()
        fig.show()
        return fig


if 'data' in globals():
    # Example 1: pass a PyG Data object directly.
    plotter = MoleculeInteractivePlotter(data=data)
    fig = plotter.show()
else:
    # Example 2: pass graph_info directly.
    graph_info = {
        'pos': [[0.0, 0.0, 0.0], [1.2, 0.0, 0.0], [0.6, 1.0, 0.0]],
        'edge_index': [[0, 1, 1, 2, 2, 0], [1, 0, 2, 1, 0, 2]],
        'x': [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0]],
        'edge_attr': [[1, 0, 0, 0]] * 6,
    }
    plotter = MoleculeInteractivePlotter(graph_info=graph_info)
    fig = plotter.show()

In [1]:
# RDKit-based SMILES -> 3D interactive molecule visualizer with bond metadata.
import numpy as np
from collections import Counter

try:
    import plotly.graph_objects as go
except ImportError as exc:
    raise ImportError('Plotly is not installed. Run: pip install plotly') from exc

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
except ImportError as exc:
    raise ImportError('RDKit is not installed. Install with: pip install rdkit') from exc


class Smiles3DInteractivePlotter:
    def __init__(self, smiles, title='Interactive Molecule from SMILES (RDKit 3D)'):
        self.smiles = smiles
        self.title = title
        self.atom_colors = {
            'H': '#E5E7EB',
            'C': '#374151',
            'N': '#2563EB',
            'O': '#DC2626',
            'F': '#14B8A6',
            'P': '#F59E0B',
            'S': '#F97316',
            'Cl': '#10B981',
            'Br': '#92400E',
            'I': '#7C3AED',
        }
        self.atom_sizes = {
            'H': 10,
            'C': 16,
            'N': 18,
            'O': 18,
            'F': 18,
            'P': 20,
            'S': 20,
            'Cl': 21,
            'Br': 22,
            'I': 24,
        }
        self.bond_colors = {
            'SINGLE': '#64748B',
            'DOUBLE': '#0EA5E9',
            'TRIPLE': '#8B5CF6',
            'AROMATIC': '#F59E0B',
            'OTHER': '#475569',
        }
        self.bond_widths = {
            'SINGLE': 4.0,
            'DOUBLE': 5.5,
            'TRIPLE': 7.0,
            'AROMATIC': 4.8,
            'OTHER': 3.5,
        }

    def _embed_3d_from_smiles(self):
        mol = Chem.MolFromSmiles(self.smiles)
        if mol is None:
            raise ValueError(f'Invalid SMILES: {self.smiles}')

        mol = Chem.AddHs(mol)
        params = AllChem.ETKDGv3()
        params.randomSeed = 42

        status = AllChem.EmbedMolecule(mol, params)
        if status != 0:
            # Fallback for hard molecules when ETKDG fails.
            status = AllChem.EmbedMolecule(mol, useRandomCoords=True, randomSeed=42)
            if status != 0:
                raise ValueError('RDKit could not generate 3D coordinates for this SMILES.')

        try:
            AllChem.UFFOptimizeMolecule(mol, maxIters=300)
        except Exception:
            # Geometry optimization can fail for some chemistries; keep embedded conformer.
            pass

        conf = mol.GetConformer()
        xyz = np.array([
            [conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y, conf.GetAtomPosition(i).z]
            for i in range(mol.GetNumAtoms())
        ], dtype=float)
        return mol, xyz

    @staticmethod
    def _bond_type_name(bond):
        if bond.GetIsAromatic():
            return 'AROMATIC'
        btype = str(bond.GetBondType())
        if btype in {'SINGLE', 'DOUBLE', 'TRIPLE'}:
            return btype
        return 'OTHER'

    def create_figure(self):
        mol, xyz = self._embed_3d_from_smiles()

        atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
        node_colors = [self.atom_colors.get(sym, '#6B7280') for sym in atom_symbols]
        node_sizes = [self.atom_sizes.get(sym, 15) for sym in atom_symbols]

        fig = go.Figure()

        bonds = list(mol.GetBonds())
        bond_type_groups = ['SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC', 'OTHER']

        for bond_type in bond_type_groups:
            xs, ys, zs, hover = [], [], [], []
            for bond in bonds:
                bname = self._bond_type_name(bond)
                if bname != bond_type:
                    continue

                i = bond.GetBeginAtomIdx()
                j = bond.GetEndAtomIdx()
                p_i, p_j = xyz[i], xyz[j]
                bond_len = float(np.linalg.norm(p_i - p_j))

                xs += [p_i[0], p_j[0], None]
                ys += [p_i[1], p_j[1], None]
                zs += [p_i[2], p_j[2], None]

                hover_text = (
                    f'Bond {bond.GetIdx()}<br>'
                    f'Atoms: {atom_symbols[i]}{i} - {atom_symbols[j]}{j}<br>'
                    f'Type: {bname}<br>'
                    f'Order: {bond.GetBondTypeAsDouble():.1f}<br>'
                    f'Aromatic: {bond.GetIsAromatic()}<br>'
                    f'In Ring: {bond.IsInRing()}<br>'
                    f'Conjugated: {bond.GetIsConjugated()}<br>'
                    f'Length: {bond_len:.3f} A'
                )
                hover += [hover_text, hover_text, '']

            if xs:
                fig.add_trace(go.Scatter3d(
                    x=xs, y=ys, z=zs,
                    mode='lines',
                    name=f'Bond: {bond_type}',
                    line=dict(
                        color=self.bond_colors.get(bond_type, '#64748B'),
                        width=self.bond_widths.get(bond_type, 3.5),
                    ),
                    hovertext=hover,
                    hoverinfo='text',
                    opacity=0.96,
                ))

        atom_hover = []
        for i, atom in enumerate(mol.GetAtoms()):
            atom_hover.append(
                f'Atom {i}<br>'
                f'Element: {atom.GetSymbol()}<br>'
                f'Atomic No: {atom.GetAtomicNum()}<br>'
                f'Degree: {atom.GetDegree()}<br>'
                f'Formal Charge: {atom.GetFormalCharge()}<br>'
                f'Hybridization: {str(atom.GetHybridization())}<br>'
                f'Aromatic: {atom.GetIsAromatic()}<br>'
                f'In Ring: {atom.IsInRing()}<br>'
                f'x: {xyz[i,0]:.3f}<br>y: {xyz[i,1]:.3f}<br>z: {xyz[i,2]:.3f}'
            )

        fig.add_trace(go.Scatter3d(
            x=xyz[:, 0],
            y=xyz[:, 1],
            z=xyz[:, 2],
            mode='markers+text',
            text=[f'{atom_symbols[i]}{i}' for i in range(len(atom_symbols))],
            textposition='top center',
            textfont=dict(size=10, color='#111827'),
            name='Atoms',
            marker=dict(
                size=node_sizes,
                color=node_colors,
                line=dict(color='#0B1220', width=1),
                opacity=0.98,
            ),
            hovertext=atom_hover,
            hoverinfo='text',
        ))

        atom_counts = Counter(atom_symbols)
        bond_counts = Counter(self._bond_type_name(b) for b in bonds)
        avg_bond = float(np.mean([
            np.linalg.norm(xyz[b.GetBeginAtomIdx()] - xyz[b.GetEndAtomIdx()])
            for b in bonds
        ])) if bonds else 0.0

        stats_lines = [
            f'SMILES: {self.smiles}',
            f'Atoms: {mol.GetNumAtoms()}',
            f'Bonds: {mol.GetNumBonds()}',
            f'Avg bond length: {avg_bond:.3f} A',
            '',
            'Atom counts:',
        ]
        for sym in sorted(atom_counts.keys()):
            stats_lines.append(f'{sym}: {atom_counts[sym]}')

        stats_lines.append('')
        stats_lines.append('Bond counts:')
        for bname in ['SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC', 'OTHER']:
            if bond_counts.get(bname, 0):
                stats_lines.append(f'{bname}: {bond_counts[bname]}')

        fig.update_layout(
            title=self.title,
            template='plotly_white',
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                aspectmode='data',
                bgcolor='rgba(238,242,247,1)',
                camera=dict(eye=dict(x=1.55, y=1.35, z=1.1)),
            ),
            paper_bgcolor='rgba(238,242,247,1)',
            plot_bgcolor='rgba(238,242,247,1)',
            legend=dict(
                x=0.01, y=0.99,
                bgcolor='rgba(255,255,255,0.9)',
                bordercolor='#CBD5E1',
                borderwidth=1,
            ),
            margin=dict(l=0, r=0, t=60, b=0),
            annotations=[dict(
                text='<br>'.join(stats_lines),
                x=0.99, y=0.02,
                xref='paper', yref='paper',
                xanchor='right', yanchor='bottom',
                showarrow=False,
                align='left',
                bordercolor='#CBD5E1',
                borderwidth=1,
                borderpad=6,
                bgcolor='rgba(255,255,255,0.94)',
                font=dict(size=12, color='#0F172A'),
            )],
        )
        return fig

    def show(self):
        fig = self.create_figure()
        fig.show()
        return fig


smiles = 'CC(=O)Oc1ccccc1C(=O)O'  # aspirin
smiles_plotter = Smiles3DInteractivePlotter(smiles=smiles)
smiles_fig = smiles_plotter.show()